## Optional: Live Database Connection (for spot checks only)
* This notebook's primary data source is the CSV exports in `data/`, loaded below. 
* This connection exists only for occasional verification against the live database.

In [3]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
user=os.getenv('user')
database=os.getenv('database')
host=os.getenv('host')
port=os.getenv('port')
password=os.getenv('password')
# format: postgresql:user:password@localhost:port/database
conn_eda=f"postgresql://{user}:{password}@{host}:{port}/{database}"
engine_eda=create_engine(conn_eda)
# Checking Connection
pd.read_sql("select count(*)as total_customers from customers;", con=engine_eda)

,total_customers
0,102016


## Dynamic CSV Loader Script
* This python script automates the process of reading multiple `CSV` files from a directory.
* Scans the `../data` directory to find all available CSV files.
* Strips the `.csv` extension to use as a clean key, and loads each file into a dictionary of pandas dataframes.

In [ ]:
files=os.listdir("../data")
print(files)
dfs={}
# Loop through every file ound in folder
for file in files:
    name=file.replace(".csv","")
    dfs[name]=pd.read_csv(f"../data/{file}")
# Acces & view one dataset for confirmation
dfs['revenue by operator']


## Inspect Dataset Quality and Summary Statistics
* Check the structural overview of the dataframe, including data types and memory usage.
* View key descriptive statistics (like mean, min, and max) for numeric columns.
* Count any missing or null values across every column to identify gaps in the data. 

In [ ]:
dfs["revenue by operator"].info()
dfs["revenue by operator"].describe()
dfs["revenue by operator"].isnull().sum()

In [ ]:
dfs["churn label"].info()
dfs["churn label"].describe()
dfs["churn label"].isnull().sum()

In [ ]:
dfs["churn timing by operator"].info()
dfs["churn timing by operator"].describe()
dfs["churn timing by operator"].isnull().sum()

In [ ]:
dfs["monthly revenue trend"].info()
dfs["monthly revenue trend"].describe()
dfs["monthly revenue trend"].isnull().sum()

In [ ]:
dfs["recharge frequency trend"].info()
dfs["recharge frequency trend"].describe()
dfs["recharge frequency trend"].isnull().sum()

In [ ]:
dfs["call derop rate by tower"].info()
dfs["call derop rate by tower"].describe()
dfs["call derop rate by tower"].isnull().sum()

In [ ]:
dfs["cohort retention"].info()
dfs["cohort retention"].describe()
dfs["cohort retention"].isnull().sum()

## Calculate Cohort Lifespan and Filter Incomplete Retention Columns
* Find the maximum date in the `cohort_month` column to establish a stable, data anchored baseline.
* Compute the total active months (`month_since_cohort`) for each subscriber group.
* Apply `.loc[]` to replace placeholder retention percentages with `NaN` for recent months where the 3-month, 6-month, or 12-month periods have not yet completed.

In [59]:
# Convert cohort_month to datetime objects
dfs["cohort retention"]["cohort_month"]=pd.to_datetime(dfs["cohort retention"]["cohort_month"])
# Confirm datatype
dfs["cohort retention"]["cohort_month"].dtype
# Calculate the difference from te maximum date of dataset and find the difference
max_date=dfs["cohort retention"]["cohort_month"].max()
months_since_cohort=(max_date-dfs["cohort retention"]["cohort_month"])
# Extract total days and convert to approximate months
days=months_since_cohort.dt.days
dfs["cohort retention"]["month_since_cohort"]=round(days/30,2)
# Use .loc[] to mask retention columns based on cohort age
dfs["cohort retention"].loc[dfs["cohort retention"]["month_since_cohort"]<12,"retained_12m"]=np.nan
dfs["cohort retention"].loc[dfs["cohort retention"]["month_since_cohort"]<6,"retained_6m"]=np.nan
dfs["cohort retention"].loc[dfs["cohort retention"]["month_since_cohort"]<3,"retained_3m"]=np.nan